# Planning results

Download the analysis data as described in the root README, or run `eval.py`.
Read the evaluation `metrics.json` files, compute success percentages per run, and report
the mean and sample standard deviation across training seeds (policy repeats for Random).
Single-model results have no error bars. Set `RESULTS_ROOT` to your evaluation output directory.


In [ ]:
from pathlib import Path
import sys

ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
            if (p / "planning/train.py").is_file())
PLANNING = ROOT / "planning"
sys.path.insert(0, str(PLANNING))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from plot_style import apply_matplotlib_style

apply_matplotlib_style()

from analysis.results import load_results, summarize

RESULTS_ROOT = PLANNING / "outputs/eval"
OUTPUT_DIR = PLANNING / "outputs/analysis"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
runs = load_results(RESULTS_ROOT)
summary = summarize(runs)
if runs.empty:
    print(f"No evaluations in {RESULTS_ROOT}. Run eval.py as described in planning/README.md.")
else:
    display(summary)
    summary.to_csv(OUTPUT_DIR / "planning_summary.csv", index=False)
    summary.to_latex(OUTPUT_DIR / "planning_summary.tex", index=False, float_format="%.3f", na_rep="—")


## Main comparison

Goal offset 25, budget 50, history 1, and λ=10 for SMWM. Error bars show sample STD
across the available seeds, with the seed count in the table above. Use five seeds for the paper protocol.


In [ ]:
METHODS = ["smwm", "sigreg", "pldm", "dino_wm", "random", "forward_only"]
LABELS = ["SMWM", "SIGReg", "PLDM", "DINO-WM", "Random", "Forward-only"]
COLORS = ["#2066a8", "#ae282c", "#7E6AAE", "#d47264", "#c7c7c7", "#8ec1da"]
ENVS = ["TwoRoom", "Reacher", "Push-T", "OGBench-Cube"]

main = summary.query("goal_offset == 25 and eval_budget == 50").copy()
main = main[main.history.isna() | main.history.eq(1)]
main = main[main.inverse_weight.isna() | main.inverse_weight.isin([0, 10])]
main = main[~main.method.eq("smwm") | (main.inverse_weight.eq(10) & main.sigreg_weight.eq(0))]
if main.duplicated(["environment", "method"]).any():
    raise ValueError("Several settings for a method remain. Select one evaluation setting for each method.")
available = [m for m in METHODS if m in set(main.method)]
if available:
    fig, ax = plt.subplots(figsize=(7.2, 2.8), layout="constrained")
    width = 0.8 / len(available)
    for i, method in enumerate(available):
        rows = main[main.method.eq(method)].set_index("environment").reindex(ENVS)
        position = np.arange(4) + (i - (len(available) - 1) / 2) * width
        index = METHODS.index(method)
        ax.bar(position, rows["mean"], width, label=LABELS[index], color=COLORS[index])
        mask = rows.n.gt(1) & rows["std"].notna()
        ax.errorbar(position[mask], rows.loc[mask, "mean"], yerr=rows.loc[mask, "std"],
                    fmt="none", ecolor="0.2", capsize=2, lw=0.8)
    ax.set(xticks=np.arange(4), xticklabels=ENVS, ylabel="Planning success (%)", ylim=(0, 105))
    ax.spines[["top", "right"]].set_visible(False)
    ax.legend(ncol=min(4, len(available)), loc="upper center", bbox_to_anchor=(0.5, 1.25))
    fig.savefig(OUTPUT_DIR / "planning_comparison.pdf", bbox_inches="tight")
    plt.show()
